# 设置Agent名称

`name` 在 Multi-Agent 场景中最常被提及，用于区分不同的 Agent。但它的作用并不局限于多 Agent 编排。在实际工程中，出现如下场景，通常都建议为 Agent 设置一个清晰且稳定的 `name`。

### 1. 流式输出归因
在启用流式输出时，`name` 可用于标识当前输出内容来自哪个 Agent。

这在多 Agent 协作、Agent 嵌套调用，或前端需要实时展示不同执行主体输出时尤其有用，便于准确区分 token 或事件的来源。

### 2. 消息身份标记
设置 `name` 后，Agent 产生的 AIMessage 会携带对应的 `name` 信息。

这使得系统在保存会话记录、回放执行过程、构建审计日志或前端展示消息角色时，能够明确识别消息的生成者。

### 3. 调试与trace可读性
在调试、日志分析和链路追踪过程中，`name` 可以作为 Agent 的稳定标识，帮助开发者快速判断当前执行的是哪个 Agent。

当系统中存在多个能力相近的 Agent，或一个 Agent 被嵌套在更复杂的工作流中时，名称能够显著提升 trace 的可读性和问题定位效率。

### 4. 组件化封装
在工程实践中，Agent 常被封装为可复用的能力模块，例如检索助手、SQL 助手、报告生成助手等。

为 Agent 设置 `name`，有助于在模块注册、运行监控、日志归档和能力复用时保持一致的身份标识。
如果后续需要将该 Agent 进一步作为子图节点、工具能力或子模块接入更复杂系统，也能降低维护和迁移成本。

### 5. 前端展示与运行态可观测性
在带有可视化界面的应用中，`name` 还可以直接作为运行时展示标识使用。

例如，在执行面板中显示“当前活跃 Agent”“本轮输出来源”或“调用链路中的执行节点”时，`name` 能帮助开发者和用户更直观地理解系统当前的执行状态。

### 6. 作为稳定的运行时身份标识
从更通用的角度看，`name` 可以理解为 Agent 在系统中的“运行时身份 ID”。

相比临时性的展示名称，一个稳定、规范的 `name` 更适合用于日志检索、监控统计、链路分析和跨模块协作，因此在生产环境中通常建议显式设置，而不是依赖默认行为。

In [1]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from rich import print as rprint

# 从.env文件中加载环境变量
load_dotenv(override=True)

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL")
)

agent = create_agent(
    model=model,
    name = "chat_assistant"
)

response = agent.invoke({"messages": ["你好"]})

rprint(response)

print("-----------------")

for msg in response["messages"]:
    msg.pretty_print()

{
    'messages': [
        HumanMessage(
            content='你好',
            additional_kwargs={},
            response_metadata={},
            id='28c17546-745b-4112-afb5-2e594e6dfc96'
        ),
        AIMessage(
            content='你好！有什么我可以帮你的吗？',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 13,
                    'prompt_tokens': 7,
                    'total_tokens': 20,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 6.375e-05,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 6.375e-05,
                        'upstream_inference_prompt_cost': 5.25e-06,
                        'upstream_inference_completions_cost': 5.85e-05
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-5.4-mini',
                'system_fingerprint': None,
                'id': 'gen-1784213998-dT6IF2yRKTzc1hWOD8lC',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            name='chat_assistant',
            id='lc_run--019f6b70-eb9e-7330-9e2c-2ee525fd8289-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 7,
                'output_tokens': 13,
                'total_tokens': 20,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        )
    ]
}

-----------------
================================ Human Message =================================

你好
================================== Ai Message ==================================
Name: chat_assistant

你好！有什么我可以帮你的吗？


# 为Agent设置系统提示词

使用 `create_agent` 创建 Agent 时，需传入**模型**和**工具**、可选地传入**系统提示词**。提示词为Agent提供了任务背景、行为准则和操作指南。

系统指令，即SystemMessage，通过 `system_prompt` 设置，定义 Agent 行为。这个参数可以是 `str` 或者 `SystemMessage` 类型。

使用建议：
- 明确说明 Agent 的角色
- 定义输出格式
- 说明何时使用工具

In [1]:
from langchain_tavily import TavilySearch
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint
import os
load_dotenv(override=True)

# 1.导入模型
model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL")
)

# 2.导入工具
web_search = TavilySearch(max_results=2)

# 3.创建Agent
agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt="你是一名多才多艺的智能助手，可以调用工具帮助用户解决问题。"
)

# 4.运行Agent获得结果
result = agent.invoke(
    {"messages": [
        {"role": "user", "content": "请帮我查询2026年足球世界杯是哪个国家举办的？"}
    ]}
)

rprint(result)


{
    'messages': [
        HumanMessage(
            content='请帮我查询2026年足球世界杯是哪个国家举办的？',
            additional_kwargs={},
            response_metadata={},
            id='51a521ee-8d7b-4370-8577-e18043b6d2a8'
        ),
        AIMessage(
            content='2026年足球世界杯由 **美国、加拿大和墨西哥** 三个国家联合举办。  \n\n这是世界杯历史上首次由 
**三个国家共同主办**，也是首次扩军到 **48支球队** 的世界杯。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 53,
                    'prompt_tokens': 1206,
                    'total_tokens': 1259,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 0.001143,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 0.001143,
                        'upstream_inference_prompt_cost': 0.0009045,
                        'upstream_inference_completions_cost': 0.0002385
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-5.4-mini',
                'system_fingerprint': None,
                'id': 'gen-1784216377-6YRPuWeNJt3fbOF3VA3o',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019f6b95-49a8-7843-b358-c8dc932dcea1-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1206,
                'output_tokens': 53,
                'total_tokens': 1259,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        )
    ]
}